# Week 12, Day 4: Stochastic gradient descent: learning from minibatches

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w012/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1951)
T, d = 5000, 20
X = rng.normal(size=(T, d)) * np.linspace(1, 0.3, d)        # features of differing scales
w_true = rng.normal(size=d)
y = X @ w_true + 0.5 * rng.normal(size=T)
w_star = np.linalg.lstsq(X, y, rcond=None)[0]
def loss(w): return 0.5 * np.mean((X @ w - y) ** 2)
L_star = loss(w_star)
print("best possible loss:", round(L_star, 4))

### Exercise: Minibatch SGD with epochs

Write `sgd(lr, batch, epochs, decay=False, seed=0)`: start from zero weights; each epoch, shuffle the indices, step through them in minibatches of size `batch` (the last may be smaller), and update with the minibatch gradient $X_{\mathcal{B}}^\top(X_{\mathcal{B}}\mathbf{w} - \mathbf{y}_{\mathcal{B}})/|\mathcal{B}|$. With `decay=True` use the learning rate lr/(1 + k/500) at update k. Return the excess loss, loss minus `L_star`, after each epoch.

In [ ]:
def sgd(lr, batch, epochs, decay=False, seed=0):
    r = np.random.default_rng(seed)
    w = np.zeros(d)
    k = 0
    history = []
    for epoch in range(epochs):
        order = r.permutation(T)
        for start in range(0, T, batch):
            b = order[start:start + batch]
            g = ...
            eta = lr / (1 + k / 500) if decay else lr
            w = ...
            k += 1
        history.append(loss(w) - L_star)
    return np.array(history)

print("excess loss per epoch, batch 10:", sgd(0.02, 10, 5).round(5))

In [ ]:
import numpy as np
def test_shape():
    "sgd returns the excess loss after each epoch"
    h = sgd(0.05, 50, 3)
    assert h.shape == (3,) and np.all(h >= -1e-12)

def test_full_batch():
    "with a batch of the whole dataset, one epoch is one exact gradient step"
    w = np.zeros(d) - 0.2 * X.T @ (X @ np.zeros(d) - y) / T
    assert np.isclose(sgd(0.2, T, 1)[0], loss(w) - L_star)

test_shape()
test_full_batch()
print("All checks passed")

### Exercise: Same computation, different batch sizes

With five epochs each (the same number of gradient evaluations), run SGD with batch sizes 1, 10, 100 and the whole dataset (T). Use learning rate 0.02 for batches below 100 and 0.2 otherwise (larger batches tolerate larger steps). Store the final excess losses in `results`, in the order of `batch_sizes`, and plot them.

In [ ]:
batch_sizes = [1, 10, 100, T]
results = []
for B in batch_sizes:
    lr = ...
    results.append(...)

plt.figure(figsize=(5, 3)); plt.loglog(batch_sizes, results, "o-"); plt.xlabel("batch size"); plt.ylabel("excess loss after 5 epochs"); plt.show()
print(dict(zip(batch_sizes, np.round(results, 5).tolist())))

In [ ]:
import numpy as np
def test_results():
    "results holds the final excess loss for each batch size after 5 epochs"
    assert len(results) == len(batch_sizes)

def test_minibatch_wins():
    "minibatches of 10 or 100 beat full-batch gradient descent by a factor of over 100 for the same five epochs"
    full = results[batch_sizes.index(T)]
    assert results[batch_sizes.index(10)] < full / 100 and results[batch_sizes.index(100)] < full / 100

test_results()
test_minibatch_wins()
print("All checks passed")

### Exercise: Constant versus decaying learning rate

With batch size 1, run 5 epochs with a constant learning rate of 0.05 (`const`), the same rate decayed as lr/(1 + k/500) (`decayed`), and a constant rate of 0.1 (`big_lr`). Plot all three excess losses against epoch on a log scale.

In [ ]:
const = ...
decayed = ...
big_lr = ...

plt.figure(figsize=(5, 3.2))
for h, lab in [(const, "constant 0.05"), (big_lr, "constant 0.1"), (decayed, "0.05 / (1 + k/500)")]:
    plt.semilogy(range(1, 6), h, "o-", label=lab)
plt.xlabel("epoch"); plt.ylabel("excess loss"); plt.legend(frameon=False); plt.show()
print("final excess loss: constant", round(const[-1], 5), " larger constant", round(big_lr[-1], 5), " decayed", round(decayed[-1], 6))

In [ ]:
import numpy as np
def test_floor():
    "the constant learning rate stalls at a higher excess loss than the decaying one, by over a factor of 10"
    assert const[-1] > 10 * decayed[-1]

def test_larger_floor():
    "a larger constant learning rate gives a higher noise floor"
    assert big_lr[-1] > const[-1]

test_floor()
test_larger_floor()
print("All checks passed")